In [16]:
import torch
import torch.nn as nn
from torch.nn import Sequential, Linear, ReLU, Sigmoid

In [39]:
class VAE(nn.Module):
    def __init__(self, input_dim, hidden_dim, latent_dim):
        super().__init__()
        self.encoder = Sequential(Linear(input_dim, hidden_dim), 
                                  ReLU())
        self.mu = Linear(hidden_dim, latent_dim)
        self.logvar = Linear(hidden_dim, latent_dim)

        self.decoder = Sequential(Linear(latent_dim, hidden_dim), ReLU(),
                                  Linear(hidden_dim, input_dim), Sigmoid())
        
    def encode(self, x):
        h = self.encoder(x)
        return self.mu(h), self.logvar(h)
    
    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)   # samples from N(0,I), same shape as std
        return mu + eps * std
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decoder(z), mu, logvar
    
def vae_loss(recon_x, x, mu, logvar, beta = 1.0):
    recon_loss = nn.functional.binary_cross_entropy(recon_x, x, reduction='sum')
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return recon_loss + beta * kl_loss

In [43]:
vae = VAE(
    input_dim=784,
    hidden_dim=400,
    latent_dim=20
)
x = torch.rand(128,784)

optimizer = torch.optim.Adam(
    vae.parameters(),
    lr=0.001
)


for epoch in range(10):

    reconstructed, mu, logvar = vae(x)

    loss = vae_loss(
        reconstructed,
        x,
        mu,
        logvar
    )

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    print(loss.item())

70407.1328125
70088.1328125
69922.6875
69855.9453125
69800.34375
69847.7421875
69816.625
69802.8984375
69818.7265625
69774.4609375


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class VAEOutput:
    z: torch.Tensor
    mu: torch.Tensor
    std: torch.Tensor
    x_recon: torch.Tensor
    loss: torch.Tensor
    loss_recon: torch.Tensor
    loss_kl: torch.Tensor

class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=512, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_std = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        # Softplus + epsilon for stable std deviation
        std = F.softplus(self.fc_std(h)) + 1e-6
        return mu, std

    def reparameterize(self, mu, std):
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x, kl_weight=1.0):
        mu, std = self.encode(x)
        z = self.reparameterize(mu, std)
        x_recon = self.decode(z)

        # 1. Reconstruction Loss (Binary Cross Entropy for MNIST)
        # Sum over features, mean over batch
        recon_loss = F.binary_cross_entropy_with_logits(x_recon, x, reduction='none').sum(dim=1).mean()

        # 2. KL Divergence
        # Analytic KL for Normal distributions
        kl_loss = -0.5 * torch.sum(1 + torch.log(std**2) - mu**2 - std**2, dim=1).mean()

        # 3. Total Loss (ELBO)
        loss = recon_loss + (kl_weight * kl_loss)

        return VAEOutput(z, mu, std, x_recon, loss, recon_loss, kl_loss)

# --- Training Loop Example ---
def train_step(model, batch, optimizer, kl_weight=1.0):
    model.train()
    optimizer.zero_grad()

    # Forward pass
    output = model(batch, kl_weight)

    # Backward pass
    output.loss.backward()

    # Gradient clipping (recommended)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    return output.loss.item()


In [37]:
import os
import numpy as np
import tensorflow as tf
import keras
from keras import ops
from keras import layers, Input, Model
from keras.layers import Dense, Layer, Flatten

In [52]:
latent_dim = 2
inputdim = 784

class Sampling(Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        # set seeds for random reprod
        self.seed_generator = keras.random.SeedGenerator(67)
    # call for (x) keras functionality
    def call(self, inputs):
        # unpack vars
        z_mean, z_log_var = inputs
        # batch is how many input images/ rows
        # dim is the encoder end dims
        batch = ops.shape(z_mean)[0]
        dim = ops.shape(z_mean)[1]
        # sample from multivariable gaussian
        epsilon = keras.random.normal(shape = (batch, dim), seed = self.seed_generator)
        # convert from log var back to exp and add
        return z_mean + ops.exp(0.5*z_log_var)*epsilon

# build encoder model
encoder_inputs = Input((inputdim,))
x = Dense(512, activation="relu")(encoder_inputs)
x = Dense(256, activation= "relu")(x)
# get means and log vars of each dim in the multivariable gaussian
z_mean = layers.Dense(latent_dim, name="z_mean")(x)
z_log_var = layers.Dense(latent_dim, name="z_log_var")(x)
z = Sampling()([z_mean, z_log_var])
encoder = Model(encoder_inputs, [z_mean, z_log_var, z], name="encoder")

# unencode everything and get sigmoid prob outputs
latent_inputs = keras.Input(shape=(latent_dim,))
x = layers.Dense(256, activation="relu")(latent_inputs)
x = layers.Dense(512, activation="relu")(x)
decoder_outputs = layers.Dense(inputdim, activation="sigmoid")(x)
decoder = Model(latent_inputs, decoder_outputs, name="decoder")

In [65]:
class VAE(Model):
    def __init__(self, encoder, decoder, **kwargs):
        # associate encoder / decoder with class VAE
        super().__init__(**kwargs)
        self.encoder = encoder
        self.decoder = decoder
        # displays
        self.total_loss_tracker = keras.metrics.Mean(name="total_loss")
        self.reconstruction_loss_tracker = keras.metrics.Mean(name="reconstruction_loss")
        self.kl_loss_tracker = keras.metrics.Mean(name = "kl_loss")

    @property
    def metrics(self):
        return [ 
            self.total_loss_tracker,
            self.reconstruction_loss_tracker,
            self.kl_loss_tracker
        ]
    
    def train_step(self, data):
        # gradient tape keeps track of gradients to do backprop
        with tf.GradientTape() as tape:
            z_mean, z_log_var, z = self.encoder(data)
            reconstruction = self.decoder(z)
            # derived losses , chose your own reconstruction loss
            reconstruction_loss = ops.mean(
                keras.losses.binary_crossentropy(data, reconstruction)
            ) * inputdim
            kl_loss = -0.5 * (1 + z_log_var - ops.square(z_mean) - ops.exp(z_log_var))
            kl_loss = ops.mean(ops.sum(kl_loss, axis=1))
            total_loss = reconstruction_loss + kl_loss
            # trainable weights comes from keras itself -> model / layer 
            grads = tape.gradient(total_loss, self.trainable_weights)
            # update gradients
            self.optimizer.apply_gradients(zip(grads, self.trainable_weights))
            # update displays
            self.total_loss_tracker.update_state(total_loss)
            self.reconstruction_loss_tracker.update_state(reconstruction_loss)
            self.kl_loss_tracker.update_state(kl_loss)
            return {
            "loss": self.total_loss_tracker.result(),
            "reconstruction_loss": self.reconstruction_loss_tracker.result(),
            "kl_loss": self.kl_loss_tracker.result(),
            }

In [ ]:
vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())
vae.fit(mnist_digits, epochs=30, batch_size=128)

In [66]:
(x_train, _), (x_test, _) = keras.datasets.mnist.load_data()
mnist_digits = np.concatenate([x_train, x_test], axis=0)
mnist_digits = mnist_digits.reshape(-1, 784).astype("float32") / 255.0
vae = VAE(encoder, decoder)
vae.compile(optimizer=keras.optimizers.Adam())
vae.fit(mnist_digits, epochs=30, batch_size=128)

Epoch 1/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 17ms/step - kl_loss: 5.0228 - loss: 183.9313 - reconstruction_loss: 178.9085
Epoch 2/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 11s 20ms/step - kl_loss: 5.1289 - loss: 161.6586 - reconstruction_loss: 156.5295
Epoch 3/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - kl_loss: 5.5687 - loss: 155.8674 - reconstruction_loss: 150.2987
Epoch 4/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 22s 35ms/step - kl_loss: 5.8452 - loss: 152.2887 - reconstruction_loss: 146.4434
Epoch 5/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 14s 25ms/step - kl_loss: 6.0190 - loss: 149.8614 - reconstruction_loss: 143.8425
Epoch 6/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 19s 34ms/step - kl_loss: 6.1370 - loss: 148.2189 - reconstruction_loss: 142.0818
Epoch 7/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 18s 33ms/step - kl_loss: 6.2195 - loss: 146.9809 - reconstruction_loss: 140.7614
Epoch 8/30
547/547 ━━━━━━━━━━━━━━━━━━━━ 12s 18ms/step - kl_loss: 6.2862 - loss: 145.8129 - reconstruction_loss: 139.5268
Epoch 9/30
547/547 ━━━━━━━━━━━━━

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

@dataclass
class VAEOutput:
    z: torch.Tensor
    mu: torch.Tensor
    std: torch.Tensor
    x_recon: torch.Tensor
    loss: torch.Tensor
    loss_recon: torch.Tensor
    loss_kl: torch.Tensor

class VAE(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=512, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh()
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_std = nn.Linear(hidden_dim, latent_dim)

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim)
        )

    def encode(self, x):
        h = self.encoder(x)
        mu = self.fc_mu(h)
        # Softplus + epsilon for stable std deviation
        std = F.softplus(self.fc_std(h)) + 1e-6
        return mu, std

    def reparameterize(self, mu, std):
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x, kl_weight=1.0):
        mu, std = self.encode(x)
        z = self.reparameterize(mu, std)
        x_recon = self.decode(z)

        # 1. Reconstruction Loss (Binary Cross Entropy for MNIST)
        # Sum over features, mean over batch
        recon_loss = F.binary_cross_entropy_with_logits(x_recon, x, reduction='none').sum(dim=1).mean()

        # 2. KL Divergence
        # Analytic KL for Normal distributions
        kl_loss = -0.5 * torch.sum(1 + torch.log(std**2) - mu**2 - std**2, dim=1).mean()

        # 3. Total Loss (ELBO)
        loss = recon_loss + (kl_weight * kl_loss)

        return VAEOutput(z, mu, std, x_recon, loss, recon_loss, kl_loss)

# --- Training Loop Example ---
def train_step(model, batch, optimizer, kl_weight=1.0):
    model.train()
    optimizer.zero_grad()

    # Forward pass
    output = model(batch, kl_weight)

    # Backward pass
    output.loss.backward()

    # Gradient clipping (recommended)
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    optimizer.step()
    return output.loss.item()
